# House Prices: Advanced Regression Techniques

Bài thực hành xây dựng các mô hình Học máy (XGBoost, Decision Tree, Random Forest, Gradient Boosting, LightGBM, Ridge Regression, PyTorch MLP)

Quy trình dữ liệu chuẩn (chống rò rỉ):

- Tập `train.csv` gốc được chia **trước tiên** thành: **Train (60%)**, **Validation (20%)**, **Internal Test (20%)**, phân tầng theo bin của `SalePrice` (`stratify`).
- Imputer (mean/mode), One-Hot Encoder và StandardScaler được `fit` **chỉ trên tập Train**, các tập còn lại chỉ `transform`.
- Tập `test.csv` giữ làm **Kaggle Test** riêng biệt để tạo submission.

## 1. Import thư viện

In [1]:
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from scipy import stats
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
import xgboost
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import wandb
import warnings
warnings.filterwarnings('ignore')

## 2. Cấu hình kịch bản

In [2]:
params = {
    "data": {
        "input_dir": r"C:\kaggle\input\house-prices-advanced-regression-techniques",
        "train_file": "train.csv",
        "test_file": "test.csv",
        "internal_test_size": 0.20,
        "val_size": 0.25,
        "random_state": 42
    },
    "experiment": {
        "expr_dir": "experiments",
        "expr_name": "house_prices_v1",
        "images_dir": "images",
        "figure_dpi": 150,
        "save_figures": True
    },
    "tuning": {
        "MLP": [
            {"name": "baseline",  "hidden1": 50, "hidden2": 25, "learning_rate": 0.01,  "batch_size": 10, "epochs": 20},
            {"name": "compact",   "hidden1": 32, "hidden2": 16, "learning_rate": 0.01,  "batch_size": 10, "epochs": 20},
            {"name": "wide",      "hidden1": 64, "hidden2": 32, "learning_rate": 0.01,  "batch_size": 10, "epochs": 20},
            {"name": "slow-long", "hidden1": 50, "hidden2": 25, "learning_rate": 0.005, "batch_size": 10, "epochs": 40},
        ],
        "XGBoost": [
            {"name": "baseline", "n_estimators": 200, "max_depth": 5, "learning_rate": 0.1},
            {"name": "deep-300", "n_estimators": 300, "max_depth": 7, "learning_rate": 0.05},
            {"name": "flat-500", "n_estimators": 500, "max_depth": 3, "learning_rate": 0.05},
            {"name": "mid-400",  "n_estimators": 400, "max_depth": 4, "learning_rate": 0.05},
        ],
        "GradientBoosting": [
            {"name": "baseline", "n_estimators": 200, "max_depth": 5, "learning_rate": 0.1},
            {"name": "slow-400", "n_estimators": 400, "max_depth": 4, "learning_rate": 0.05},
            {"name": "deep-150", "n_estimators": 150, "max_depth": 7, "learning_rate": 0.1},
            {"name": "flat-600", "n_estimators": 600, "max_depth": 3, "learning_rate": 0.05},
        ],
    },
    "wandb": {
        "mode": "online",     # "disabled" để tắt log W&B
        "project": "house-prices-regression"
    },
    "train": {
        "hidden1": 50,
        "hidden2": 25,
        "learning_rate": 0.01,
        "batch_size": 10,
        "epochs": 20
    }
}
try:
    base = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base = os.getcwd()
input_dir = params["data"]["input_dir"]
expr_name = params["experiment"]["expr_name"]
timestamp = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")
algo_folders = {}
exp_base = os.path.join(base, params["experiment"]["expr_dir"], expr_name)
os.makedirs(exp_base, exist_ok=True)
algorithms = ['XGBoost', 'DecisionTree', 'RandomForest', 'GradientBoosting', 'LightGBM', 'RidgeRegression', 'PyTorch_MLP']
for algo in algorithms:
    folder = os.path.join(exp_base, f'{algo}_{timestamp}')
    os.makedirs(folder, exist_ok=True)
    os.makedirs(os.path.join(folder, 'images'), exist_ok=True)
    algo_folders[algo] = folder
    with open(os.path.join(folder, 'params.json'), 'w', encoding='utf-8') as f:
        json.dump(params, f, ensure_ascii=False, indent=4)
print("=" * 70)
print("BẮT ĐẦU THÍ NGHIỆM")
print(f"Thời gian        : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Thư mục thí nghiệm: {exp_base}")
for algo in algorithms:
    print(f"  {algo}: {algo_folders[algo]}")
print("=" * 70)

BẮT ĐẦU THÍ NGHIỆM
Thời gian        : 2026-09-26 15:34:44
Thư mục thí nghiệm: C:\Users\ASUS\sgu26_deep_learning\Tuần 02\experiments\house_prices_v1
  XGBoost: C:\Users\ASUS\sgu26_deep_learning\Tuần 02\experiments\house_prices_v1\XGBoost_2026-09-26_15-34-44
  DecisionTree: C:\Users\ASUS\sgu26_deep_learning\Tuần 02\experiments\house_prices_v1\DecisionTree_2026-09-26_15-34-44
  RandomForest: C:\Users\ASUS\sgu26_deep_learning\Tuần 02\experiments\house_prices_v1\RandomForest_2026-09-26_15-34-44
  GradientBoosting: C:\Users\ASUS\sgu26_deep_learning\Tuần 02\experiments\house_prices_v1\GradientBoosting_2026-09-26_15-34-44
  LightGBM: C:\Users\ASUS\sgu26_deep_learning\Tuần 02\experiments\house_prices_v1\LightGBM_2026-09-26_15-34-44
  RidgeRegression: C:\Users\ASUS\sgu26_deep_learning\Tuần 02\experiments\house_prices_v1\RidgeRegression_2026-09-26_15-34-44
  PyTorch_MLP: C:\Users\ASUS\sgu26_deep_learning\Tuần 02\experiments\house_prices_v1\PyTorch_MLP_2026-09-26_15-34-44


## 3. Load Data

In [3]:
train = pd.read_csv(os.path.join(input_dir, params["data"]["train_file"]))
test = pd.read_csv(os.path.join(input_dir, params["data"]["test_file"]))
print(f"Train Shape: {train.shape}, Test Shape: {test.shape}")
train.info()
print("\nGiá trị thiếu trong train:")
print(train.isnull().sum())

Train Shape: (1460, 81), Test Shape: (1459, 80)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  Over

## 4. Data Analysis

In [4]:
# Shape, dtypes, and basic stats
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
train.info()
test.info()
print("\n--- Target Distribution ---")
print(train["SalePrice"].describe())
print(f"Skewness: {train[chr(83)+chr(97)+chr(108)+chr(101)+chr(80)+chr(114)+chr(105)+chr(99)+chr(101)].skew():.4f}")
print(f"Kurtosis: {train['SalePrice'].kurt():.4f}")

Train shape: (1460, 81)
Test shape: (1459, 80)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  Overa

In [5]:
# Correlation with SalePrice
corr = train.select_dtypes(include=["number"]).corr()
top_corr = corr["SalePrice"].sort_values(ascending=False)
print("Top 10 features correlated with SalePrice:")
print(top_corr.head(11))
plt.figure(figsize=(12, 8))
sns.heatmap(train.select_dtypes(include=["number"]).corr(), annot=False, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(os.path.join(algo_folders["PyTorch_MLP"], "images", "correlation_heatmap.png"), dpi=150, bbox_inches="tight")
plt.close()

Top 10 features correlated with SalePrice:
SalePrice       1.000000
OverallQual     0.790982
GrLivArea       0.708624
GarageCars      0.640409
GarageArea      0.623431
TotalBsmtSF     0.613581
1stFlrSF        0.605852
FullBath        0.560664
TotRmsAbvGrd    0.533723
YearBuilt       0.522897
YearRemodAdd    0.507101
Name: SalePrice, dtype: float64


In [6]:
# SalePrice distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("Distribution of SalePrice")
stats.probplot(train["SalePrice"], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot of SalePrice")
plt.tight_layout()
plt.savefig(os.path.join(algo_folders["PyTorch_MLP"], "images", "saleprice_distribution.png"), dpi=150, bbox_inches="tight")
plt.close()

In [7]:
# Missing values
missing_train = train.isnull().sum().sort_values(ascending=False)
missing_train = missing_train[missing_train > 0]
print("Missing values in train:")
print(missing_train)
missing_test = test.isnull().sum().sort_values(ascending=False)
missing_test = missing_test[missing_test > 0]
print("\nMissing values in test:")
print(missing_test)
plt.figure(figsize=(10, 6))
sns.heatmap(train.isnull(), cbar=False, cmap="viridis")
plt.title("Missing Data Heatmap (Train)")
plt.tight_layout()
plt.savefig(os.path.join(algo_folders["PyTorch_MLP"], "images", "missing_heatmap.png"), dpi=150, bbox_inches="tight")
plt.close()

Missing values in train:
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageYrBlt       81
GarageCond        81
GarageType        81
GarageFinish      81
GarageQual        81
BsmtFinType2      38
BsmtExposure      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

Missing values in test:
PoolQC          1456
MiscFeature     1408
Alley           1352
Fence           1169
MasVnrType       894
FireplaceQu      730
LotFrontage      227
GarageYrBlt       78
GarageQual        78
GarageFinish      78
GarageCond        78
GarageType        76
BsmtCond          45
BsmtQual          44
BsmtExposure      44
BsmtFinType1      42
BsmtFinType2      42
MasVnrArea        15
MSZoning           4
BsmtHalfBath       2
Utilities          2
Functional         2
BsmtFullBath       2
BsmtFinSF1         1
BsmtFinSF2         1
BsmtUnfSF    

In [8]:
if params["experiment"]["save_figures"]:
    plt.figure(figsize=(10, 6))
    sns.heatmap(train.isnull(), cbar=False, cmap='viridis')
    plt.title("Biểu đồ thể hiện dữ liệu thiếu trong Train")
    plt.savefig(os.path.join(algo_folders['PyTorch_MLP'], 'images', 'missing_data.png'), dpi=150, bbox_inches='tight')
    plt.close()

## 5. Split → Preprocessing (fit trên Train)

Chia **trước** tiền xử lý, phân tầng theo 10 bin của `SalePrice` (`pd.qcut` + `stratify`):

| Tập | Tỷ lệ | Số mẫu | Vai trò |
|-----|-------|--------|----------|
| Train | 60% | 876 | fit imputer/encoder/scaler |
| Validation | 20% | 292 | giám sát khi train |
| Internal Test | 20% | 292 | đánh giá độc lập |
| Kaggle Test | — | 1.459 | submission |

Pipeline `ColumnTransformer`, `fit` **chỉ** trên `X_tr`:

- Số: `SimpleImputer(mean)` → `StandardScaler`
- Phân loại: `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown='ignore')`
- Kết quả: **262 đặc trưng**; mean SalePrice giữa các tập xấp xỉ nhau (181k / 180k / 180k)

In [9]:
drop_cols = ['Id', 'Alley', 'PoolQC', 'Fence', 'MiscFeature']
y_all = train['SalePrice'].astype(np.float32)
X_all = train.drop(columns=['SalePrice'] + drop_cols)
df_test_kaggle = test.drop(columns=drop_cols)
kaggle_ids = test['Id']
# 1) Chia tập TRƯỚC, phân tầng theo bin của SalePrice để phân phối giá trải đều các tập
price_bins_all = pd.qcut(y_all, q=10, labels=False, duplicates='drop')
X_train_full, X_internal_test, y_train_full, y_internal_test = train_test_split(
    X_all, y_all,
    test_size=params["data"]["internal_test_size"],
    random_state=params["data"]["random_state"],
    stratify=price_bins_all,
)
price_bins_full = pd.qcut(y_train_full, q=10, labels=False, duplicates='drop')
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=params["data"]["val_size"],
    random_state=params["data"]["random_state"],
    stratify=price_bins_full,
)
# 2) Pipeline tiền xử lý: imputer + scaler + one-hot
#    fit CHỈ trên tập Train để chống rò rỉ (Validation/Internal Test/Kaggle Test chỉ transform)
num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_tr.columns if c not in num_cols]
preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat_cols),
    ],
    remainder='drop',
)
preprocess.fit(X_tr)
X_tr_p = preprocess.transform(X_tr)
X_val_p = preprocess.transform(X_val)
X_internal_test_p = preprocess.transform(X_internal_test)
X_kaggle_p = preprocess.transform(df_test_kaggle)
X_tr_t = torch.tensor(X_tr_p, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr.values.astype(np.float32)).unsqueeze(1)
X_val_t = torch.tensor(X_val_p, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values.astype(np.float32)).unsqueeze(1)
X_internal_test_t = torch.tensor(X_internal_test_p, dtype=torch.float32)
X_kaggle_test_t = torch.tensor(X_kaggle_p, dtype=torch.float32)
n_feat = X_tr_p.shape[1]
print(f"Số đặc trưng sau encoding: {n_feat}")
print(
    f"Tập Train ({len(X_tr)} mẫu, 60%): {X_tr_p.shape} | "
    f"Validation ({len(X_val)} mẫu, 20%): {X_val_p.shape} | "
    f"Internal Test ({len(X_internal_test)} mẫu, 20%): {X_internal_test_p.shape}"
)
print(f"Mean SalePrice theo tập: "
      f"Train={y_tr.mean():.0f}, Val={y_val.mean():.0f}, Test={y_internal_test.mean():.0f}")

Số đặc trưng sau encoding: 262
Tập Train (876 mẫu, 60%): (876, 262) | Validation (292 mẫu, 20%): (292, 262) | Internal Test (292 mẫu, 20%): (292, 262)
Mean SalePrice theo tập: Train=181254, Val=180145, Test=180698


## 6. Define Model

In [10]:
class HouseMLP(nn.Module):
    def __init__(self, input_dim, hidden1=50, hidden2=25):
        super(HouseMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(),
            nn.Linear(hidden1, hidden2), nn.ReLU(),
            nn.Linear(hidden2, hidden1), nn.ReLU(),
            nn.Linear(hidden1, 1)
        )
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
    def forward(self, x):
        return self.network(x)
class RMSELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
    def forward(self, y_pred, y_true):
        return torch.sqrt(self.mse(y_pred, y_true) + 1e-8)
model = HouseMLP(input_dim=X_tr_p.shape[1], hidden1=params["train"]["hidden1"], hidden2=params["train"]["hidden2"])
print(model)

HouseMLP(
  (network): Sequential(
    (0): Linear(in_features=262, out_features=50, bias=True)
    (1): ReLU()
    (2): Linear(in_features=50, out_features=25, bias=True)
    (3): ReLU()
    (4): Linear(in_features=25, out_features=50, bias=True)
    (5): ReLU()
    (6): Linear(in_features=50, out_features=1, bias=True)
  )
)


## 7. Fit Model

In [11]:
# Khởi tạo W&B run cho toàn bộ thí nghiệm
# mode lấy từ params["wandb"]["mode"]: "disabled" (mặc định) hoặc "online"
run = wandb.init(
    project=params["wandb"]["project"],
    name=f"house-prices-{timestamp}",
    config=params["train"],
    mode=params["wandb"]["mode"],
)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\ASUS\_netrc.


wandb: Currently logged in as: huynhnhatminh20102005 (huynhnhatminh20102005-sai-gon-university-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u1ea7' in position 152: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users

wandb: setting up run 6niawqt8


wandb: Tracking run with wandb version 0.30.0


wandb: Run data is saved locally in C:\Users\ASUS\sgu26_deep_learning\Tuần 02\wandb\run-20260926_153455-6niawqt8
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run house-prices-2026-09-26_15-34-44


wandb:  View project at https://wandb.ai/huynhnhatminh20102005-sai-gon-university-/house-prices-regression


wandb:  View run at https://wandb.ai/huynhnhatminh20102005-sai-gon-university-/house-prices-regression/runs/6niawqt8


In [12]:
print("\n--- Huấn luyện các mô hình ---")
# XGBoost
xgb_model = xgboost.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
xgb_model.fit(X_tr_p, y_tr)
pickle.dump(xgb_model, open(os.path.join(algo_folders['XGBoost'], 'xgb_model.pkl'), 'wb'))
print("-> Hoàn tất huấn luyện XGBoost")
# Decision Tree
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_tr_p, y_tr)
print("-> Hoàn tất huấn luyện Decision Tree")
# Random Forest
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=4)
rf_model.fit(X_tr_p, y_tr)
print("-> Hoàn tất huấn luyện Random Forest")
# Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gb_model.fit(X_tr_p, y_tr)
print("-> Hoàn tất huấn luyện Gradient Boosting")
# LightGBM
lgbm_model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_depth=10, random_state=42)
lgbm_model.fit(X_tr_p, y_tr)
print("-> Hoàn tất huấn luyện LightGBM")
# Ridge Regression
ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(X_tr_p, y_tr)
print("-> Hoàn tất huấn luyện Ridge Regression")
# PyTorch MLP
torch.manual_seed(42)
model = HouseMLP(input_dim=X_tr_p.shape[1], hidden1=params["train"]["hidden1"], hidden2=params["train"]["hidden2"])
train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=params["train"]["batch_size"], shuffle=True)
criterion = RMSELoss()
optimizer = optim.Adamax(model.parameters(), lr=params["train"]["learning_rate"])
epochs = params["train"]["epochs"]
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t).numpy().flatten()
        val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    avg_loss = total_loss / len(train_loader)
    wandb.log({"epoch": epoch + 1, "train_loss": avg_loss, "val_rmse": val_rmse})
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} - Loss: {avg_loss:.4f} - Val RMSE: {val_rmse:.4f}")
torch.save({"model_state_dict": model.state_dict()}, os.path.join(algo_folders['PyTorch_MLP'], 'best_model.pt'))
print("-> Hoàn tất huấn luyện PyTorch MLP")


--- Huấn luyện các mô hình ---


-> Hoàn tất huấn luyện XGBoost
-> Hoàn tất huấn luyện Decision Tree


-> Hoàn tất huấn luyện Random Forest


-> Hoàn tất huấn luyện Gradient Boosting


-> Hoàn tất huấn luyện LightGBM
-> Hoàn tất huấn luyện Ridge Regression


Epoch  10/20 - Loss: 25111.2213 - Val RMSE: 28446.1867


Epoch  20/20 - Loss: 21541.6462 - Val RMSE: 24831.9523
-> Hoàn tất huấn luyện PyTorch MLP


## 8. Model Evaluation & Accuracy

Đánh giá trên **Internal Test (292 mẫu, 20%)** — tập chưa từng tham gia `fit` bất kỳ transform nào.

Sắp xếp theo **RMSLE** (thang điểm Kaggle), tham số phụ theo R²:

| Hạng | Model | RMSLE | R² Score | RMSE ($) | MAE ($) |
|------|-------|-------|----------|----------|----------|
| 1 | **PyTorch_MLP** | **0.11287** | **0.9095** | **24,082** | **15,771** |
| 2 | GradientBoosting | 0.12653 | 0.8818 | 27,528 | 16,054 |
| 3 | XGBoost | 0.13526 | 0.8747 | 28,341 | 16,575 |
| 4 | RandomForest | 0.13754 | 0.8870 | 26,914 | 17,082 |
| 5 | LightGBM | 0.13994 | 0.8774 | 28,040 | 17,239 |
| 6 | RidgeRegression | 0.15227 | 0.8698 | 28,886 | 19,127 |
| 7 | DecisionTree | 0.22851 | 0.6558 | 46,978 | 28,201 |

- **RMSLE** = điểm Kaggle (thấp hơn tốt hơn) → chọn model theo cột này, không theo R².
- Lưu ý thứ hạng R² và RMSLE khác nhau: RandomForest R² cao hơn GradientBoosting nhưng RMSLE tệ hơn.
- Chi tiết xem `metrics.csv` trong từng thư mục mô hình.

In [13]:
print("\n--- Đánh giá trên Internal Test ---")
model.eval()
with torch.no_grad():
    pred_mlp = model(X_internal_test_t).numpy().flatten()

preds_eval = {
    'XGBoost': xgb_model.predict(X_internal_test_p),
    'DecisionTree': dt_model.predict(X_internal_test_p),
    'RandomForest': rf_model.predict(X_internal_test_p),
    'GradientBoosting': gb_model.predict(X_internal_test_p),
    'LightGBM': lgbm_model.predict(X_internal_test_p),
    'RidgeRegression': ridge_model.predict(X_internal_test_p),
    'PyTorch_MLP': pred_mlp,
}

def rmsle(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(np.clip(y_pred, 0, None)))))

eval_results = pd.DataFrame([
    {
        "Model": name,
        "RMSE": round(float(np.sqrt(mean_squared_error(y_internal_test, p))), 2),
        "MAE": round(float(mean_absolute_error(y_internal_test, p)), 2),
        "R² Score": round(float(r2_score(y_internal_test, p)), 4),
        "RMSLE": round(rmsle(y_internal_test, p), 5),
    }
    for name, p in preds_eval.items()
])

eval_results = eval_results.sort_values("RMSLE").reset_index(drop=True)
print("\n=== BẢNG ĐÁNH GIÁ ĐỘC LẬP (Internal Test, sắp theo RMSLE - thang điểm Kaggle) ===")
print(eval_results.to_string(index=False))
print("\n=== BẢNG ACCURACY (R² Score) ===")
print(eval_results[["Model", "R² Score"]].to_string(index=False))
print("\nRMSLE thấp nhất = model tốt nhất theo thang điểm Kaggle")
# Save metrics to each folder
for algo in algorithms:
    row = eval_results[eval_results['Model'] == algo]
    if len(row) > 0:
        row.to_csv(os.path.join(algo_folders[algo], 'metrics.csv'), index=False)
# Gửi bảng đánh giá và cột điểm lên W&B
wandb.log({
    "eval/rmsle_best": float(eval_results["RMSLE"].min()),
    "eval/r2_best": float(eval_results["R² Score"].max()),
})
wandb.log({"eval_table": wandb.Table(dataframe=eval_results.reset_index(drop=True))})
wandb.finish()


--- Đánh giá trên Internal Test ---

=== BẢNG ĐÁNH GIÁ ĐỘC LẬP (Internal Test, sắp theo RMSLE - thang điểm Kaggle) ===
           Model     RMSE      MAE  R² Score   RMSLE
     PyTorch_MLP 24081.93 15770.71    0.9095 0.11287
GradientBoosting 27528.49 16054.06    0.8818 0.12653
         XGBoost 28340.94 16574.52    0.8747 0.13526
    RandomForest 26914.20 17081.72    0.8870 0.13754
        LightGBM 28039.84 17238.83    0.8774 0.13994
 RidgeRegression 28885.82 19127.03    0.8698 0.15227
    DecisionTree 46977.81 28201.99    0.6558 0.22851

=== BẢNG ACCURACY (R² Score) ===
           Model  R² Score
     PyTorch_MLP    0.9095
GradientBoosting    0.8818
         XGBoost    0.8747
    RandomForest    0.8870
        LightGBM    0.8774
 RidgeRegression    0.8698
    DecisionTree    0.6558

RMSLE thấp nhất = model tốt nhất theo thang điểm Kaggle


wandb: uploading artifact run-6niawqt8-eval_table; updating run metadata


wandb: uploading artifact run-6niawqt8-eval_table


wandb: uploading artifact run-6niawqt8-eval_table; uploading history steps 0-21, summary


wandb: uploading artifact run-6niawqt8-eval_table


wandb: uploading media\table\eval_table_21_cb736527282af1788439.table.json; uploading wandb-summary.json; uploading config.yaml


wandb: uploading wandb-summary.json; uploading config.yaml


wandb: 
wandb: Run history:
wandb:           epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb:    eval/r2_best ▁
wandb: eval/rmsle_best ▁
wandb:      train_loss █▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        val_rmse █▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:           epoch 20
wandb:    eval/r2_best 0.9095
wandb: eval/rmsle_best 0.11287
wandb:      train_loss 21541.64617
wandb:        val_rmse 24831.95232
wandb: 


wandb:  View run house-prices-2026-09-26_15-34-44 at: https://wandb.ai/huynhnhatminh20102005-sai-gon-university-/house-prices-regression/runs/6niawqt8
wandb:  View project at: https://wandb.ai/huynhnhatminh20102005-sai-gon-university-/house-prices-regression
wandb: Synced 4 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260926_153455-6niawqt8\logs


## 9. Tinh chỉnh siêu tham số (Tuning)

Chọn cấu hình tốt nhất **trên tập Validation (292 mẫu)** — không dùng Internal Test để tránh rò rỉ chọn model.

Mỗi cấu hình = **1 W&B run riêng** (`mode` lấy từ `params["wandb"]["mode"]`). Tiêu chí chọn: **RMSLE thấp nhất**. Cấu hình thắng được dùng lại ở §11 (train full).

| Model | Cấu hình tốt nhất | Val RMSLE | Baseline Val RMSLE |
|-------|-------------------|-----------|--------------------|
| PyTorch MLP | `wide`: hidden 64/32, lr=0.01, epochs=20 | **0.12828** | 0.13969 |
| XGBoost | `flat-500`: 500 trees, depth=3, lr=0.05 | **0.13537** | 0.14834 |
| Gradient Boosting | `flat-600`: 600 trees, depth=3, lr=0.05 | **0.13684** | 0.14126 |

Tổng 12 cấu hình (4×MLP, 4×XGBoost, 4×GradientBoosting) — cả 3 model đều **cải thiện** so với baseline. Kết quả đầy đủ tại `experiments/house_prices_v1/tuning_results.csv`. 4 model còn lại (Random Forest, LightGBM, Ridge, Decision Tree) giữ tham số mặc định.

In [14]:
print("\n--- Tuning siêu tham số (chấm trên Validation 292 mẫu) ---")

def val_rmsle(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(np.log1p(y_true),
                                            np.log1p(np.clip(y_pred, 0, None)))))

tuning_rows = []

# 1) PyTorch MLP
for cfg in params["tuning"]["MLP"]:
    torch.manual_seed(42)
    m = HouseMLP(input_dim=X_tr_p.shape[1], hidden1=cfg["hidden1"], hidden2=cfg["hidden2"])
    dl = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=cfg["batch_size"], shuffle=True)
    crit = RMSELoss()
    opt = optim.Adamax(m.parameters(), lr=cfg["learning_rate"])
    run = wandb.init(project=params["wandb"]["project"],
                     name=f"tuning-MLP-{cfg['name']}", config=cfg,
                     mode=params["wandb"]["mode"])
    for ep in range(cfg["epochs"]):
        m.train()
        for xb, yb in dl:
            opt.zero_grad()
            loss = crit(m(xb), yb)
            loss.backward()
            opt.step()
    m.eval()
    with torch.no_grad():
        pv = m(X_val_t).numpy().flatten()
    r = val_rmsle(y_val, pv)
    wandb.log({"val_rmsle": r})
    wandb.finish()
    tuning_rows.append({"Model": "PyTorch_MLP", "Config": cfg["name"],
                        "hidden1": cfg["hidden1"], "hidden2": cfg["hidden2"],
                        "learning_rate": cfg["learning_rate"],
                        "batch_size": cfg["batch_size"], "epochs": cfg["epochs"],
                        "n_estimators": np.nan, "max_depth": np.nan,
                        "Val RMSLE": round(r, 5)})
    print(f"  MLP {cfg['name']:12s} -> Val RMSLE {r:.5f}")

# 2) XGBoost + 3) Gradient Boosting
for name, builder in [
    ("XGBoost", lambda c: xgboost.XGBRegressor(
        n_estimators=int(c["n_estimators"]), max_depth=int(c["max_depth"]),
        learning_rate=c["learning_rate"], random_state=42)),
    ("GradientBoosting", lambda c: GradientBoostingRegressor(
        n_estimators=int(c["n_estimators"]), max_depth=int(c["max_depth"]),
        learning_rate=c["learning_rate"], random_state=42)),
]:
    for cfg in params["tuning"][name]:
        m = builder(cfg)
        run = wandb.init(project=params["wandb"]["project"],
                         name=f"tuning-{name}-{cfg['name']}", config=cfg,
                         mode=params["wandb"]["mode"])
        m.fit(X_tr_p, y_tr)
        r = val_rmsle(y_val, m.predict(X_val_p))
        wandb.log({"val_rmsle": r})
        wandb.finish()
        tuning_rows.append({"Model": name, "Config": cfg["name"],
                            "hidden1": np.nan, "hidden2": np.nan,
                            "learning_rate": cfg["learning_rate"],
                            "batch_size": np.nan, "epochs": np.nan,
                            "n_estimators": int(cfg["n_estimators"]),
                            "max_depth": int(cfg["max_depth"]),
                            "Val RMSLE": round(r, 5)})
        print(f"  {name} {cfg['name']:12s} -> Val RMSLE {r:.5f}")

tuning_df = (pd.DataFrame(tuning_rows)
             .sort_values(["Model", "Val RMSLE"])
             .reset_index(drop=True))
print("\n=== BẢNG TUNING (sắp theo Model, Val RMSLE) ===")
print(tuning_df.to_string(index=False))
tuning_df.to_csv(os.path.join(exp_base, "tuning_results.csv"), index=False)
print(f"\nĐã lưu tuning_results.csv -> {exp_base}")

# Cấu hình thắng cho từng model (dùng lại ở §11)
best_params = {}
for mdl in tuning_df["Model"].unique():
    row = tuning_df[tuning_df["Model"] == mdl].iloc[0]
    best_params[mdl] = {k: row[k] for k in
                        ["hidden1", "hidden2", "learning_rate", "batch_size",
                         "epochs", "n_estimators", "max_depth"]
                        if k in tuning_df.columns and pd.notna(row[k])}

run = wandb.init(project=params["wandb"]["project"], name="tuning-summary",
                 config=params["tuning"], mode=params["wandb"]["mode"])
wandb.log({"tuning_table": wandb.Table(dataframe=tuning_df)})
wandb.finish()

print("\nCấu hình tốt nhất theo Validation (dùng cho §11):")
for mdl, cfg in best_params.items():
    print(f"  {mdl:16s} {cfg}")


--- Tuning siêu tham số (chấm trên Validation 292 mẫu) ---


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u1ea7' in position 152: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users

wandb: setting up run loph2p1m


wandb: Tracking run with wandb version 0.30.0


wandb: Run data is saved locally in C:\Users\ASUS\sgu26_deep_learning\Tuần 02\wandb\run-20260926_153517-loph2p1m
wandb: Run `wandb offline` to turn off syncing.


...

wandb: uploading requirements.txt; uploading media\table\tuning_table_0_89a9cec1482f552d7c24.table.json; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json


wandb: uploading history steps 0-0, summary


wandb:  View run tuning-summary at: https://wandb.ai/huynhnhatminh20102005-sai-gon-university-/house-prices-regression/runs/s1c93cxa
wandb:  View project at: https://wandb.ai/huynhnhatminh20102005-sai-gon-university-/house-prices-regression
wandb: Synced 4 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260926_153725-s1c93cxa\logs



Cấu hình tốt nhất theo Validation (dùng cho §11):
  GradientBoosting {'learning_rate': 0.05, 'n_estimators': 600.0, 'max_depth': 3.0}
  PyTorch_MLP      {'hidden1': 64.0, 'hidden2': 32.0, 'learning_rate': 0.01, 'batch_size': 10.0, 'epochs': 20.0}
  XGBoost          {'learning_rate': 0.05, 'n_estimators': 500.0, 'max_depth': 3.0}


## 10. Prediction & Submission

In [15]:
submission_df = pd.read_csv(os.path.join(base, 'sample_submission.csv'))
# Predict on Kaggle Test
model.eval()
with torch.no_grad():
    pred_mlp_k = model(X_kaggle_test_t).numpy().flatten()
preds_kaggle = {
    'XGBoost': xgb_model.predict(X_kaggle_p),
    'DecisionTree': dt_model.predict(X_kaggle_p),
    'RandomForest': rf_model.predict(X_kaggle_p),
    'GradientBoosting': gb_model.predict(X_kaggle_p),
    'LightGBM': lgbm_model.predict(X_kaggle_p),
    'RidgeRegression': ridge_model.predict(X_kaggle_p),
    'PyTorch_MLP': pred_mlp_k,
}
print("Hoàn tất dự đoán cho tập Test Kaggle!")
# Save all submissions
for algo, pred in preds_kaggle.items():
    sub = submission_df.copy()
    sub['SalePrice'] = pred
    sub.to_csv(os.path.join(algo_folders[algo], 'submission.csv'), index=False)
print(f"Đã xuất submission cho {len(preds_kaggle)} mô hình (model huấn luyện trên 60% Train)")
print("7 file submission full nằm trong từng thư mục mô hình: <Model>_<timestamp>/submission_<Model>_full.csv")

Hoàn tất dự đoán cho tập Test Kaggle!


Đã xuất submission cho 7 mô hình (model huấn luyện trên 60% Train)
7 file submission full nằm trong từng thư mục mô hình: <Model>_<timestamp>/submission_<Model>_full.csv


## 11. Final Models cho Kaggle

Các mô hình ở trên chỉ huấn luyện trên **876 mẫu (60%)** để đánh giá trung thực trên Internal Test.

Cell này train lại **toàn bộ 7 thuật toán trên 1.460 mẫu** (Kaggle tự chấm trên test riêng nên không phát sinh rò rỉ), rồi xuất **7 file submission riêng lẻ** — mỗi file đúng 1 thuật toán:

| File | Thuật toán |
|------|------------|
| `submission_PyTorch_MLP_full.csv` | PyTorch MLP (RMSLE nội bộ 0.11287) |
| `submission_GradientBoosting_full.csv` | Gradient Boosting (0.12653) |
| `submission_XGBoost_full.csv` | XGBoost (0.13526) |
| `submission_RandomForest_full.csv` | Random Forest (0.13754) |
| `submission_LightGBM_full.csv` | LightGBM (0.13994) |
| `submission_RidgeRegression_full.csv` | Ridge Regression (0.15227) |
| `submission_DecisionTree_full.csv` | Decision Tree (0.22851) |

Mỗi file ghi vào **thư mục của chính model đó**: `experiments/house_prices_v1/<Model>_<timestamp>/`. Ngoài ra MLP/XGBoost/GradientBoosting dùng **cấu hình thắng ở §9 Tuning**. Lưu ý: điểm Kaggle là **RMSLE — thấp hơn tốt hơn**.

In [16]:
print("\n--- Final models: train trên toàn bộ 1.460 mẫu, tách riêng từng thuật toán ---")

X_full = train.drop(columns=['SalePrice'] + drop_cols)
y_full = train['SalePrice'].astype(np.float32)
df_test_raw = test.drop(columns=drop_cols)

num_cols_f = X_full.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_f = [c for c in X_full.columns if c not in num_cols_f]

prep_final = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
    ]), num_cols_f),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), cat_cols_f),
])
X_full_p = prep_final.fit_transform(X_full)
X_kaggle_final = prep_final.transform(df_test_raw)

full_preds = {}

# Cấu hình thắng từ §9 Tuning (fallback về params["train"] nếu chưa chạy tuning)
bp_all = globals().get("best_params", {})
bp_mlp = bp_all.get("PyTorch_MLP", {})
bp_xgb = bp_all.get("XGBoost", {})
bp_gb = bp_all.get("GradientBoosting", {})

# (a) PyTorch MLP trên full data với cấu hình đã tune
torch.manual_seed(42)
mlp_final = HouseMLP(input_dim=X_full_p.shape[1],
                     hidden1=int(bp_mlp.get("hidden1", params["train"]["hidden1"])),
                     hidden2=int(bp_mlp.get("hidden2", params["train"]["hidden2"])))
xt_full = torch.tensor(X_full_p, dtype=torch.float32)
yt_full = torch.tensor(y_full.values.astype(np.float32)).unsqueeze(1)
full_loader = DataLoader(TensorDataset(xt_full, yt_full),
                            batch_size=int(bp_mlp.get("batch_size", params["train"]["batch_size"])), shuffle=True)
crit_final = RMSELoss()
opt_final = optim.Adamax(mlp_final.parameters(),
                         lr=float(bp_mlp.get("learning_rate", params["train"]["learning_rate"])))
epochs_final = int(bp_mlp.get("epochs", 30))
for ep in range(epochs_final):
    mlp_final.train()
    for xb, yb in full_loader:
        opt_final.zero_grad()
        loss = crit_final(mlp_final(xb), yb)
        loss.backward()
        opt_final.step()
mlp_final.eval()
with torch.no_grad():
    full_preds['PyTorch_MLP'] = mlp_final(torch.tensor(X_kaggle_final, dtype=torch.float32)).numpy().flatten()
print(f"  -> PyTorch_MLP: train full {len(X_full)} mẫu, {epochs_final} epoch OK")

# (b) 6 thuật toán còn lại train trên full data
full_models = {
    'XGBoost': xgboost.XGBRegressor(
        n_estimators=int(bp_xgb.get("n_estimators", 200)),
        max_depth=int(bp_xgb.get("max_depth", 5)),
        learning_rate=float(bp_xgb.get("learning_rate", 0.1)), random_state=42),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=4),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=int(bp_gb.get("n_estimators", 200)),
        max_depth=int(bp_gb.get("max_depth", 5)),
        learning_rate=float(bp_gb.get("learning_rate", 0.1)), random_state=42),
    'LightGBM': HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_depth=10, random_state=42),
    'RidgeRegression': Ridge(alpha=1.0, random_state=42),
}
for name, m in full_models.items():
    m.fit(X_full_p, y_full)
    full_preds[name] = m.predict(X_kaggle_final)
    print(f"  -> {name}: train full {len(X_full)} mẫu OK")

# (c) Xuất submission RIÊNG LẺ cho từng thuật toán (mỗi file = 1 model)
sub_full = pd.read_csv(os.path.join(base, 'sample_submission.csv'))
for name, p in full_preds.items():
    s = sub_full.copy()
    s['SalePrice'] = p
    path = os.path.join(algo_folders[name], f'submission_{name}_full.csv')
    s.to_csv(path, index=False)
    print(f"    {name:16s} -> submission_{name}_full.csv")

print(f"\nĐã ghi {len(full_preds)} file submission riêng lẻ (mỗi file = 1 thuật toán, full 1.460 mẫu)")
print("Nằm trong từng thư mục mô hình: <Model>_<timestamp>/submission_<Model>_full.csv")
if bp_all:
    print("Đã dùng cấu hình thắng từ §9 Tuning:", list(bp_all.keys()))
else:
    print("Chưa chạy tuning -> dùng cấu hình mặc định")


--- Final models: train trên toàn bộ 1.460 mẫu, tách riêng từng thuật toán ---


  -> PyTorch_MLP: train full 1460 mẫu, 20 epoch OK


  -> XGBoost: train full 1460 mẫu OK
  -> DecisionTree: train full 1460 mẫu OK


  -> RandomForest: train full 1460 mẫu OK


  -> GradientBoosting: train full 1460 mẫu OK


  -> LightGBM: train full 1460 mẫu OK
  -> RidgeRegression: train full 1460 mẫu OK
    PyTorch_MLP      -> submission_PyTorch_MLP_full.csv
    XGBoost          -> submission_XGBoost_full.csv
    DecisionTree     -> submission_DecisionTree_full.csv
    RandomForest     -> submission_RandomForest_full.csv
    GradientBoosting -> submission_GradientBoosting_full.csv
    LightGBM         -> submission_LightGBM_full.csv
    RidgeRegression  -> submission_RidgeRegression_full.csv

Đã ghi 7 file submission riêng lẻ (mỗi file = 1 thuật toán, full 1.460 mẫu)
Nằm trong từng thư mục mô hình: <Model>_<timestamp>/submission_<Model>_full.csv
Đã dùng cấu hình thắng từ §9 Tuning: ['GradientBoosting', 'PyTorch_MLP', 'XGBoost']


## Kết luận

Quy trình thực hiện: Load Data → Data Analysis → Split (stratify theo bin SalePrice) → Preprocessing fit trên Train → Define Model → Fit Model → Model Evaluation → **Tuning (§9, chọn theo Val RMSLE)** → Prediction → Submission

Chống rò rỉ: imputer/one-hot/scaler chỉ `fit` trên tập Train (60% = 876 mẫu), Validation và Internal Test chỉ `transform`.

Kết quả trên Internal Test (run `2026-09-26_15-34-44`, sắp theo RMSLE):

| Model | RMSLE | R² Score | RMSE ($) | MAE ($) |
|-------|-------|----------|----------|----------|
| **PyTorch_MLP** | **0.11287** | **0.9095** | **24,082** | **15,771** |
| GradientBoosting | 0.12653 | 0.8818 | 27,528 | 16,054 |
| XGBoost | 0.13526 | 0.8747 | 28,341 | 16,575 |
| RandomForest | 0.13754 | 0.8870 | 26,914 | 17,082 |
| LightGBM | 0.13994 | 0.8774 | 28,040 | 17,239 |
| RidgeRegression | 0.15227 | 0.8698 | 28,886 | 19,127 |
| DecisionTree | 0.22851 | 0.6558 | 46,978 | 28,201 |

**Submission (cell §11):** 7 file `experiments/house_prices_v1/<Model>_<timestamp>/submission_<Model>_full.csv` — mỗi file = 1 thuật toán train full 1.460 mẫu, không trộn lẫn.

**Tuning (§9, chọn theo Val RMSLE):** MLP `wide` 64/32 → 0.12828 · XGBoost 500/depth3/lr0.05 → 0.13537 · GradientBoosting 600/depth3/lr0.05 → 0.13684 — cấu hình thắng dùng ở cell §11.

**Best Model (Internal Test): PyTorch_MLP — RMSLE 0.11287, R² 0.9095**

## Kết thúc
Thời gian: 2026-09-26 15:34:44
Artifacts: Tuần 02/experiments/house_prices_v1/*_2026-09-26_15-34-44/ (submission_<Model>_full.csv nằm trong từng thư mục model) + tuning_results.csv
W&B: https://wandb.ai/huynhnhatminh20102005-sai-gon-university-/house-prices-regression (mode=online)